In [ ]:
import papermill as pm
import subprocess, os, time
from joblib import Parallel, delayed

In [ ]:
py_nb = 'WaterfowlHabitatSinglefips_recalc_v2.ipynb'
pm.inspect_notebook(py_nb)

In [ ]:
# import duckdb
# con = duckdb.connect()
# con.install_extension("spatial")
# con.load_extension("spatial")
# con.install_extension("azure")
# con.load_extension("azure")

In [ ]:
# nwiurl = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\abdu_cn_east_wet_wkb.parquet"
nwiurl = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\abdu_cn_east_weth2o.parquet"
hucsurl = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\abdu_cn_east_wsheds.parquet"
# waterurl = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\old_abdu_openH2o_cn_east_wkb.parquet"
waterurl = True

In [ ]:
# aoi = [970495.21943597, 2173156.98696821, 1066128.90295111, 2294692.19188825] #'{4706305B-A25D-48AD-9876-242887FEA92D}' 
params = dict(
    pid_fld = 'dgr_blk',
    nwiurl = nwiurl,
    wetattrfld = 'CLASS_NAME',
    waterurl = waterurl,
    hucsurl = hucsurl,
    hucidfld = 'WATERSHED_CODE',
    crossWalk_json = 'https://giscog.blob.core.windows.net/abdu/caWetlands.json',
    nrgy_csv = 'azure://abdu/ehjv_kcal.csv',
    protLands = r"D:\ABDUBounds\ABDU_Canada_Data\abdu_ca_protLands.parquet",
    urbanMask = r"D:\ABDUBounds\abdu_ca_urban.parquet",
    demand = 'azure://abdu/Demand9Species_Merged.parquet',
    in_crs = {p:'ESRI:102008' for p in (nwiurl,hucsurl) if not p is None}
)

In [ ]:
# con.sql('''CREATE SECRET (
#     TYPE AZURE,
#     ACCOUNT_NAME 'giscog')''')
# con.sql("SET azure_transport_option_type = 'curl'")
# con.sql(f"""
# CREATE OR REPLACE TABLE counties AS
# SELECT NAME, STATE_NAME, FIPS, geometry FROM read_parquet('azure://abdu/uscounties.parquet')
# WHERE STATE_NAME = 'Mississippi'
# """)

In [ ]:
# allfips = con.sql('select distinct(fips) from counties').df().values.tolist()

In [ ]:
# listaoi = sorted([item for items in allfips for item in items])
# print(listaoi)

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Geometry is in a geographic CRS.")
warnings.filterwarnings("ignore", message="IProgress not found.")

In [ ]:
import geopandas as gpd
from shapely import Polygon
# f = "D:/ABDUBounds/ca_prov.parquet"
# df = gpd.read_parquet(f, filters=[["postal", "in", ['ON', 'NB', 'NL', 'NS', 'PE', 'QC']]])
df = gpd.read_parquet(params['hucsurl'])
df = df.to_crs(4326)
coords = df.geometry.get_coordinates()
polys = []
x = int(coords.x.min())-1
y = int(coords.y.min())-1
while y < int(coords.y.max())+1:
    poly = Polygon(((x, y),
        (x, y+1),
        (x+1, y+1),
        (x+1, y)
        ))
    if df.geometry.centroid.within(poly, align=False).any():
        polys.append(poly)
    x += 1
    if x == int(coords.x.max())+1:
        x = int(coords.x.min())-1
        y += 1
s = gpd.GeoSeries(polys,
        crs=4326)
print(len(s))
ax1 = df.plot(color='white', edgecolor='black')
s.boundary.plot(ax=ax1)

In [ ]:
coords = s.to_crs(5070).get_coordinates()
listaoi = [coords.loc[i,:].values.tolist() for i in coords.index.unique()]
df = gpd.GeoDataFrame(dict(fname='', pcs5070=map(str,listaoi), geometry=s), crs=4326)
df.loc[:,'fname'] = df.geometry.centroid.apply(lambda pt: f"n{int(pt.y)}_e{int(pt.x)}".replace('e-','w').replace('n-','s'))
print(len(listaoi))
df.info()

In [ ]:
df.fname.is_unique, df.pcs5070.is_unique

In [ ]:
def f(x):
    if isinstance(x, str):
        fname = x
    else:
        # pt = s.to_crs(4326).centroid[listaoi.index(x)]
        # fname = f"n{int(pt.y)}_e{int(pt.x)}".replace('e-','w').replace('n-','s')
        fname = df[df.pcs5070==str(x)].fname.values[0]
    try:         
        if os.path.isfile('./output/'+fname+'.parquet'):
            return (fname, 'exists')
        else:
            params['aoi'] = x
            subprocess.run(pm.execute_notebook(py_nb,'./output/{0}.ipynb'.format(fname),parameters=params), shell=True)
            return (fname, 'run complete')
    except Exception as e:
        if os.path.isfile('./output/'+fname+'.parquet'):
            return (fname, 'run complete')
        return (fname,'fail', e)

In [ ]:
start_time = time.time()
print(time.ctime(start_time))

In [ ]:
completed = Parallel(n_jobs=2, verbose=10)(delayed(f)(listaoi[x]) for x in range(len(listaoi)))

In [ ]:
print('Done in {0:.1f} seconds'.format(time.time() - start_time))

In [ ]:
len([i for i in completed if not i[1] == 'run complete'])
len([i for i in completed if not i[1] in ('run complete', 'exists')])
# [i for i in completed if not i[1] == 'run complete']

In [ ]:
#check for failed
completed
#fixme = completed

In [ ]:
hucs = gpd.read_parquet(params['hucsurl'])[params['hucidfld']].to_list()
print(len(hucs))
n0 = 0; hucs_done = []
for i in os.listdir('./output/'):
    if i.endswith('.parquet') and i.startswith('n'):
        if n0==0:
            cols = gpd.read_parquet(f'./output/{i}').columns.to_list()
        n, c = gpd.read_parquet(f'./output/{i}').shape
        n0 += n
        hucs_done.extend(gpd.read_parquet(f'./output/{i}')[params['hucidfld']].to_list())
        print(i, (n,c), n0)
        print(set(cols) - set(gpd.read_parquet(f'./output/{i}').columns.to_list()))
        print()
    

In [ ]:
len(set(hucs) - set(hucs_done)),set(hucs) - set(hucs_done)

In [ ]:
#merge all parquet files in ./output/*.parquet
# if not 'duckdb' in locals().keys():
#     print('importing duckdb')
#     import duckdb
#     duckdb.load_extension("spatial")
# # fname = os.path.basename(params['hucsurl']).replace('.parquet', 'wH2Ocalc') #.rsplit('\\',1)[1]
# fname = 'abdu_cn_east_v2'
# print(fname)
# duckdb.execute(f"""
# COPY (SELECT * FROM './output/n*.parquet') TO './output/{fname}.parquet' (FORMAT 'parquet');
# """)

In [ ]:
# listaoi = df[df.fname.isin(['n44_w76', 'n49_w95', 'n50_w95'])].geometry.to_crs(5070).bounds.values.tolist()
# listaoi = df[df.fname.isin(['n45_w61', 'n46_w61', 'n46_w60'])].geometry.to_crs(5070).bounds.values.tolist()
coords = df[df.fname.isin(['n47_w69','n49_w64'])].to_crs(5070).get_coordinates()
listaoi = [coords.loc[i,:].values.tolist() for i in coords.index.unique()]
listaoi

In [ ]:
for fname in df.fname:
    if os.path.isfile('./output/'+fname+'.ipynb'):
        print(fname, 'ipynb exists')
        if os.path.isfile('./output/'+fname+'.parquet'):
            print(fname, 'parquet exists')   
        else:
            os.remove(('./output/'+fname+'.ipynb'))
            print('removed ipynb')